# LLM Eval - Testing PromptRegistry & LLMTracer

This notebook tests the `prompts.py` and `tracer.py` modules.

In [ ]:
import sys
sys.path.insert(0, '..')

In [ ]:
from llm_eval.prompts import PromptRegistry
from llm_eval.tracer import LLMTracer
from llm_eval.store import TraceStore

## 1. Test PromptRegistry

In [ ]:
# Initialize registry (creates/uses test_registry.db)
registry = PromptRegistry("test_registry.db")
print("Registry initialized!")

In [ ]:
# Register first version of a prompt
prompt_v1 = registry.register(
    name="summarizer",
    template="You are a helpful assistant. Summarize the following text: {text}",
    description="Initial summarizer prompt"
)

print(f"Registered: {prompt_v1.name} v{prompt_v1.version}")
print(f"Hash: {prompt_v1.template_hash[:16]}...")

In [ ]:
# Register same template again - should return existing (dedup)
prompt_same = registry.register(
    name="summarizer",
    template="You are a helpful assistant. Summarize the following text: {text}",
    description="This description is ignored since template is same"
)

print(f"Same template returned: v{prompt_same.version} (should be v1)")
assert prompt_same.version == 1, "Should return existing version!"

In [ ]:
# Register new version with different template
prompt_v2 = registry.register(
    name="summarizer",
    template="You are an expert summarizer. Create a concise, accurate summary of: {text}",
    description="Improved prompt with expert framing"
)

print(f"New version: {prompt_v2.name} v{prompt_v2.version}")
assert prompt_v2.version == 2, "Should be version 2!"

In [ ]:
# List all versions
versions = registry.list_versions("summarizer")
print(f"Found {len(versions)} versions:")
for v in versions:
    print(f"  v{v.version}: {v.description}")

In [ ]:
# Get specific version
v1 = registry.get("summarizer", version=1)
print(f"Got v1: {v1.template[:50]}...")

# Get latest (default)
latest = registry.get("summarizer")
print(f"Latest is v{latest.version}")

In [ ]:
# Test KeyError for non-existent prompt
try:
    registry.get("nonexistent")
except KeyError as e:
    print(f"Correctly raised KeyError: {e}")

## 2. Test LLMTracer (without actual LLM call)

In [ ]:
# Initialize tracer
tracer = LLMTracer(
    db_path="test_registry.db",  # Same DB as registry
    project="test_project",
    session_id="test_session_001"
)

print(f"Tracer initialized!")
print(f"  Project: {tracer.project}")
print(f"  Session: {tracer.session_id}")

In [ ]:
# Test new_session
new_sess = tracer.new_session()
print(f"New session (auto UUID): {new_sess}")

named_sess = tracer.new_session("my_experiment_001")
print(f"Named session: {named_sess}")

In [ ]:
# Test set_prompt_context
tracer.set_prompt_context("summarizer", version=1)
print(f"Prompt context set to: {tracer._prompt_name} v{tracer._prompt_version}")

In [ ]:
# Test error when prompt not set
tracer2 = LLMTracer(db_path="test_registry.db", project="test")
from uuid import uuid4

try:
    tracer2.on_llm_start(
        serialized={},
        prompts=["test"],
        run_id=uuid4()
    )
except RuntimeError as e:
    print(f"Correctly raised RuntimeError: {e}")

## 3. Test LLMTracer with Mock LLM Call

In [ ]:
from uuid import uuid4
from unittest.mock import MagicMock
import time

# Create a fresh tracer
tracer = LLMTracer(
    db_path="test_registry.db",
    project="mock_test",
    session_id="mock_session_001"
)
tracer.set_prompt_context("summarizer", version=2)

# Simulate on_llm_start
run_id = uuid4()
tracer.on_llm_start(
    serialized={"name": "ChatOpenAI"},
    prompts=[],
    run_id=run_id,
    messages=[
        {"role": "system", "content": "You are an expert summarizer."},
        {"role": "user", "content": "Summarize: The quick brown fox jumps over the lazy dog."}
    ]
)
print(f"on_llm_start called with run_id: {run_id}")
print(f"Pending calls: {len(tracer._pending)}")

In [ ]:
# Simulate some processing time
time.sleep(0.1)

# Create mock LLMResult
mock_generation = MagicMock()
mock_generation.text = "A fox jumps over a dog."
mock_generation.message = MagicMock()
mock_generation.message.content = "A fox jumps over a dog."
mock_generation.message.usage_metadata = {
    "input_tokens": 25,
    "output_tokens": 8
}
mock_generation.message.response_metadata = {
    "model_name": "gpt-4"
}

mock_response = MagicMock()
mock_response.generations = [[mock_generation]]
mock_response.llm_output = {
    "token_usage": {
        "prompt_tokens": 25,
        "completion_tokens": 8,
        "total_tokens": 33
    },
    "model_name": "gpt-4"
}

# Call on_llm_end
tracer.on_llm_end(response=mock_response, run_id=run_id)
print("on_llm_end called")
print(f"Pending calls after end: {len(tracer._pending)}")

In [ ]:
# Verify trace was saved
store = TraceStore("test_registry.db")
traces = store.get_traces({"session_id": "mock_session_001"})

print(f"Found {len(traces)} trace(s):")
for t in traces:
    print(f"  ID: {t['id'][:8]}...")
    print(f"  Prompt: {t['prompt_name']} v{t['prompt_version']}")
    print(f"  Output: {t['output_content']}")
    print(f"  Tokens: {t['total_tokens']}")
    print(f"  Latency: {t['latency_ms']}ms")
    print(f"  Status: {t['status']}")

## 4. Test Error Handling

In [ ]:
# Test on_llm_error
tracer.new_session("error_test_session")
tracer.set_prompt_context("summarizer")

error_run_id = uuid4()
tracer.on_llm_start(
    serialized={},
    prompts=["Test prompt"],
    run_id=error_run_id
)

# Simulate error
time.sleep(0.05)
tracer.on_llm_error(
    error=Exception("API rate limit exceeded"),
    run_id=error_run_id
)

print("Error trace saved")

In [ ]:
# Verify error trace
error_traces = store.get_traces({"session_id": "error_test_session"})
print(f"Error traces: {len(error_traces)}")
for t in error_traces:
    print(f"  Status: {t['status']}")
    print(f"  Error: {t['error']}")

## 5. View All Sessions

In [ ]:
sessions = store.get_sessions()
print(f"Total sessions: {len(sessions)}\n")
for s in sessions:
    print(f"Session: {s['session_id']}")
    print(f"  Traces: {s['trace_count']}")
    print(f"  Success: {s['success_count']}, Errors: {s['error_count']}")
    print(f"  Total tokens: {s['total_tokens']}")
    print()

## Cleanup

In [ ]:
# Uncomment to delete test database
# import os
# os.remove("test_registry.db")
# print("Deleted test_registry.db")